In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch_geometric

import sys
sys.path.append('./..')

from ld_gcn import network, loader, plotting, preprocessing, testing, error, training, utils, initialization, loss
from IPython.display import HTML

import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
pde_problem = 14
problem_name, variable, mu_space, n_param, dim_pde, n_comp, n_sim, HyperParams = utils.prepare_HyperParams(pde_problem)
device = initialization.initialize(HyperParams)
with open('../dataset/coanda_unstructured/index.list', 'r') as file:
    viscosities = file.readlines()

viscosities = [float(mu) for mu in viscosities]
viscosities = np.array([viscosities[i] for i in range(0, len(viscosities))])
mu_space = viscosities

In [ ]:
from torch_geometric.data import Dataset
from scipy.io import loadmat


import numpy as np

def prolongate_trajectory(snapshot_matrix, start_time=1, end_time=81):
    """
    start time: absolute time of start
    end_time: end time
    """
    start_idx = start_time - 1
    end_idx   = end_time - 1

    current_length = snapshot_matrix.shape[1]

    if current_length > end_idx:
        return snapshot_matrix[:, start_idx:end_idx + 1]

    last_snapshot = snapshot_matrix[:, -1:]  # shape [num_dofs, 1]
    num_missing = end_time - current_length

    if num_missing > 0:
        prolongated_part = np.repeat(last_snapshot, num_missing, axis=1)
        extended = np.concatenate((snapshot_matrix, prolongated_part), axis=1)
    else:
        extended = snapshot_matrix

    return extended[:, start_idx:end_idx + 1]


class LoadDataset(Dataset):
    def __init__(self, root_dir, variable, dim_pde, n_comp, mu_space, end_time = 10, start_time=6):
        # variable should be "U1" in this case
        # x ha shape [num DoFs, 1]
        self.U1 = []
        self.U2 = []

        for component in (1, 2):
            variable = f'U{component}'
            for i, param in enumerate(mu_space):
                self.append_traj(param, root_dir, i, component, variable, start_time, end_time)
            
        self.VX = torch.tensor(np.concatenate(self.U1, axis=1), dtype=torch.float32)
        self.VY = torch.tensor(np.concatenate(self.U2, axis=1), dtype=torch.float32)
        # self.U = np.stack([self.U1, self.U2], axis=-1)
        # self.U = torch.tensor(self.U, dtype=torch.float32)
        self.dim = dim_pde
        self.n_comp = n_comp

        self.xx = self.xx.repeat(1, self.VX.shape[1])
        self.yy = self.yy.repeat(1, self.VX.shape[1])

    def append_traj(self, param, root_dir, i, component, variable, start_time, end_time):
        param_name = str(param)
        file = root_dir + param_name + f'/mat_NS/u{component}_snapshots_mu_{param_name}.mat'
        data_mat = loadmat(file)
        tmp = data_mat[variable]
        tmp = prolongate_trajectory(tmp, end_time=end_time, start_time=start_time)
        attribute = self.U1 if component == 1 else self.U2
        attribute.append(tmp)
        if i == 0:
            self.xx = torch.tensor(data_mat['xx'], dtype=torch.float32)
            self.yy = torch.tensor(data_mat['yy'], dtype=torch.float32)
            self.T = torch.tensor(data_mat['T'].astype(int)) # it should have shape [something, 3]
            self.E = torch.tensor(data_mat['E'].astype(int)) # it should have shape [something, 2]

    def len(self):
        pass
    
    def get(self):
        pass

In [ ]:
end_time = 120
start_time = 8
dataset_dir = '../dataset/'+problem_name+'_unstructured/mu/'
dataset = LoadDataset(dataset_dir, variable, dim_pde, n_comp, mu_space, end_time=end_time, start_time=start_time)

In [ ]:
def create_param_list(mu_space, start_time, end_time, device):
    mu_space = np.array(mu_space)       # shape [n_mu]
    n_mu = len(mu_space)

    times = np.arange(start_time, end_time + 1)    # shape [n_times]. Absolute times are included
    n_times = len(times)

    mu_expanded = mu_space[:, None].repeat(n_times, axis=1)
    t_expanded = times[None, :].repeat(n_mu, axis=0)
    params = np.stack([mu_expanded, t_expanded], axis=-1)  # [n_mu, n_times, 2]

    return torch.tensor(params, dtype=torch.float32, device=device)

params = create_param_list(mu_space, start_time=start_time, end_time=end_time, device=device)
n_sim = params.shape[0]
n_snap2keep = params.shape[1]
delete_first_n = 0
#dataset, mu_space = preprocessing.delete_initial_condition(dataset, mu_space, n_comp, n_snap2keep, n_delete=delete_first_n, shrink_param_space=True)

In [ ]:
graph_loader, train_loader, test_loader, \
    val_loader, scaler_all, scaler_test, xyz, VAR_all, VAR_val, VAR_test, \
        train_trajectories, val_trajectories, test_trajectories, params_train, params_test = preprocessing.graphs_dataset(dataset, HyperParams, params)

In [ ]:
RecNet = network.RecNet(HyperParams)
RecNet = RecNet.to(device)
DynNet = network.DynNet(HyperParams)
DynNet = DynNet.to(device)

torch.set_default_dtype(torch.float32)

optimizer = 'ADAM' # 'ADAM' or 'LBFGS'

if optimizer == 'ADAM':
    optimizer = torch.optim.Adam([
        {'params': DynNet.parameters()},
        {'params': RecNet.parameters()}
      ],
      lr=HyperParams.learning_rate,
      weight_decay=HyperParams.weight_decay
    )

elif optimizer == 'LBFGS':
    optimizer = torch.optim.LBFGS(
        list(DynNet.parameters())+list(RecNet.parameters()),
        lr = 1.,
        max_iter = HyperParams.max_epochs,
        max_eval = None,
        tolerance_grad = 1e-07,
        tolerance_change = 1e-09,
        history_size = 30,
        line_search_fn = 'strong_wolfe',
    )
    
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=HyperParams.miles, gamma=HyperParams.gamma)

In [ ]:
compile = False

if compile:
    try:
        import torch._dynamo
        torch._dynamo.config.suppress_errors = True
        RecNet = torch.compile(RecNet)
        DynNet = torch.compile(DynNet)
        print('The networks have been compiled successfully')
    except:
        print('Not possible to compile the networks')

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters in dynnet:", count_trainable_params(DynNet))
print("Trainable parameters in recnet:", count_trainable_params(RecNet))

# To reduce memory consumption on GPU:
params = params.to("cpu")
VAR_all = VAR_all.to("cpu")
VAR_val = VAR_val.to("cpu")
VAR_test = VAR_test.to("cpu")

if device=='cuda':
    torch.cuda.empty_cache()

In [ ]:
load = True
train = False

if load:
    try:
        RecNet.load_state_dict(torch.load(HyperParams.net_dir+HyperParams.net_name+HyperParams.net_run+'_decoder.pt', map_location=torch.device('cpu')))
        DynNet.load_state_dict(torch.load(HyperParams.net_dir+HyperParams.net_name+HyperParams.net_run+'_dyn.pt', map_location=torch.device('cpu')))
        print('Loading saved network')
    except FileNotFoundError:
        print('Not possible to load the network')
        train = True

if train:
    training.train(RecNet, DynNet, optimizer, device, scheduler, train_loader, test_loader, HyperParams, params_train, params_test, loss.physics_loss)

In [ ]:
RecNet = RecNet.to("cpu")
DynNet = DynNet.to("cpu")

vars = "GCA-ROM"
VAR_train = VAR_all[train_trajectories,:,:]

results, latents = testing.evaluate(VAR_all, RecNet, DynNet, graph_loader, params, HyperParams)
results_test, latents_test = testing.evaluate(VAR_test, RecNet, DynNet, test_loader, params_test, HyperParams)
results_train, latents_ = testing.evaluate(VAR_train, RecNet, DynNet, train_loader, params_train, HyperParams)
results_val = results[val_trajectories,:,:]

In [ ]:
vars = problem_name

error_abs, norm = error.compute_error(results, VAR_all, scaler_all)
error_abs_test, norm_test = error.compute_error(results_test, VAR_test, scaler_all)
error_abs_train, norm_train = error.compute_error(results_train, VAR_train, scaler_all)
error_abs_val, norm_val = error.compute_error(results_val, VAR_val, scaler_all)

print('\nERRORS ON THE TRAINING SET:')
error.print_error(error_abs_train, norm_train, vars)
print('\nERRORS ON THE TEST SET:')
error.print_error(error_abs_test, norm_test, vars)
print('\nERRORS ON THE VALIDATION DATASET:')
error.print_error(error_abs_val, norm_val, vars)
print('\nERRORS ON THE WHOLE DATASET:')
error.print_error(error_abs, norm, vars)

error.save_error(error_abs_test, norm_test, HyperParams, vars)

In [ ]:
n_snapshots = n_snap2keep - delete_first_n

max_error_index = np.argmax(np.array(error_abs)/np.array(norm))
print(f'Index of the max of the relative error: {max_error_index}')
print(f'Indices of test trajectories: {(np.array(test_trajectories)/(n_snapshots))[::n_snapshots]}')

In [ ]:
plotting.plot_loss(HyperParams)

In [ ]:
SAMPLE = 0
plotting.plot_latent_time(HyperParams, SAMPLE, latents, params, n_sim)

In [ ]:
def plot_latent_component(HyperParams, component, latents, params, param_sample):
    """
    This function plots the evolution of latent states over time and saves the plot as a .png file.

    Parameters:
    latents (np.ndarray): The latent states.
    params (list): The parameters.
    HyperParams (object): The hyperparameters.
    param_sample (int): The number of simulations.

    Returns:
    None
    """

    plt.figure()
    sequence_length = latents.shape[0] // param_sample # basically the length of the time integration
    cmap = cm.get_cmap('viridis')
    colors = [cmap(i / (params.shape[0] - 1)) for i in range(params.shape[0])]

    times = np.array(np.arange(sequence_length))*HyperParams.dt+ params[...,-1][0][0].item()
    
    for SAMPLE in range(params.shape[0]):
        start = SAMPLE * sequence_length # SAMPLE probably refers to the SAMPLE-th parameter in the sequence
        end = start + sequence_length # end of the integration

        if SAMPLE == 0:
            label = fr'$\mu={params[0,0,0].item():.2f}$'
        elif SAMPLE == params.shape[0] - 1:
            label = rf'$\mu={params[-1,0,0].item():.2f}$'
        else:
            label = None

        stn_evolution = latents[start:end, component]
        plt.plot(times, stn_evolution.detach().numpy(), label=label, color=colors[SAMPLE])

    plt.xlabel('$t$')
    plt.legend(loc='upper right')
    plt.ylabel(f'$s_{component+1}(t)$')
    #plt.title(f'Latent state evolution for component {component}')
    plt.grid(True, which="both", ls="--", alpha=0.1)
    plt.savefig(HyperParams.net_dir+f'latent_component_{component}'+HyperParams.net_run+str(SAMPLE)+'.pdf', bbox_inches='tight')#, dpi=500)
    plt.show()

for component in {0, 1}:
    plot_latent_component(HyperParams, component, latents, params, n_sim)

In [ ]:
total_times = n_snap2keep - delete_first_n
SAMPLE = max_error_index//total_times # index of the parameter for which we want to plot, in the case the one with the greatest error
SNAP = max_error_index - SAMPLE*total_times #SNAP represents the i-th snapshot in the time evolution

plotting.plot_fields(SAMPLE, SNAP, results, scaler_all, HyperParams, dataset, params, lid_driven=True)
#plotting.plot_fields(19, 70, results, scaler_all, HyperParams, dataset, params, lid_driven = True)
plotting.plot_fields(0, params.shape[1]-1, results, scaler_all, HyperParams, dataset, params, True)

In [ ]:
final_time = params_train[..., -1][0][-1].to(device='cpu')
rel_errors = np.array(error_abs)/np.array(norm)
rel_errors_test = np.array(error_abs_test)/np.array(norm_test)

plotting.plot_relative_errors_vs_time(rel_errors, params, HyperParams, final_time, flag='all')

In [ ]:
def find_dofs_on_symmetry_axis(yy, value, tol=1e-5):
    axis_dofs = []
    for i in range(yy.shape[0]):
        if abs(yy[i,0]-value) < tol:
            axis_dofs.append(i)
    return axis_dofs

on_axis = find_dofs_on_symmetry_axis(dataset.yy, value=3.75, tol=1e-5)
#for dof in on_axis:
#    print(f'DOF index: {dof}, x-coordinate: {dataset.xx[dof,0].item()}')

P_diagram = on_axis[4] # point P for the bifurcation diagram
print(f"P=({dataset.xx[P_diagram, 0].item()}, {dataset.yy[P_diagram, 0].item()})")

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.labelsize': 22,  #'x-large',
    'legend.fontsize': 19,
    'xtick.labelsize': 17,
    'ytick.labelsize': 17,
    'figure.titlesize': 23
})


def plot_bifurcation_diagram(HyperParams, results, VAR_all, mu_space, scaler_all, P_diagram):
    z_net = preprocessing.inverse_normalize_input(results.permute(1,0,2), scaler_all)
    ground_truth = preprocessing.inverse_normalize_input(VAR_all.permute(1,0,2), scaler_all)
    bifurcation_values_net = []
    bifurcation_values_gt = []
    bifurcation_norms_net = []
    bifurcation_norms_gt = []
    n_times = z_net.shape[1] // len(mu_space)

    for i in range(len(mu_space)):
        current_index = i * n_times + (n_times - 1)
        snapshot_net = z_net[P_diagram, current_index, 1].item()
        snapshot_gt = ground_truth[P_diagram, current_index, 1].item()
        norm_net = torch.norm(z_net[:, current_index, 1]).item()
        norm_gt = torch.norm(ground_truth[:, current_index, 1]).item()
        bifurcation_values_net.append(snapshot_net)
        bifurcation_values_gt.append(snapshot_gt)
        bifurcation_norms_net.append(norm_net)
        bifurcation_norms_gt.append(norm_gt)

    bifurcation_values_net, bifurcation_values_gt = np.array(bifurcation_values_net), np.array(bifurcation_values_gt)

    plt.plot(mu_space, bifurcation_values_gt, color='#065895',label=r'$u_{2,h}(P,T)$', marker='o')
    plt.plot(mu_space, bifurcation_values_net, color='#F79A25', label=r'$u_{2, \text{sim}}(P,T)$', marker='o')
    plt.xlabel(r'$\mu$')#, fontsize=18)
    #plt.ylabel(r'$u_2(P, T)$')
    #plt.title('Bifurcation Diagram at Point P')
    plt.legend()
    plt.grid(True, which="both", ls="--", color='gray', alpha=0.1)  
    plt.savefig(HyperParams.net_dir+'bifurcation_'+HyperParams.net_run+'_point'+'.pdf', bbox_inches='tight')
    plt.show()

    plt.plot(mu_space, bifurcation_norms_gt, color='#065895', label=r'$\|\boldsymbol{u}_{2, h}(T)\|$', marker='o')
    plt.plot(mu_space, bifurcation_norms_net, color='#F79A25', label=r'$\|\boldsymbol{u}_{2,\text{sim}}(T)\|$', marker='o')
    plt.xlabel(r'$\mu$')#, fontsize=18)
    #plt.ylabel(r'$\|u_2(T)\|$')
    #plt.title('Bifurcation Diagram at Point P')
    plt.legend()
    plt.grid(True, which="both", ls="--", color='gray', alpha=0.1)
    plt.savefig(HyperParams.net_dir+'bifurcation_'+HyperParams.net_run+f'_norm'+'.pdf', bbox_inches='tight')
    plt.show()

def plot_latent_bifurcation_diagram(HyperParams, latents, mu_space, component=0, plot_norm = False):
    n_times = latents.shape[0] // len(mu_space)
    latent_values = []
    latent_norms = []
    latent_widths = []

    for i in range(len(mu_space)):
        current_index = i * n_times + (n_times - 1)
        latent_snapshot = latents[current_index, component].item()
        latent_norm = torch.norm(latents[current_index, :]).item()
        latent_width = torch.max(latents[i*(n_times):(i+1)*n_times, component]) - torch.min(latents[i*(n_times):(i+1)*n_times, component]).item()
        latent_values.append(latent_snapshot)
        latent_norms.append(latent_norm)
        latent_widths.append(latent_width)
    latent_values = np.array(latent_values)

    plt.plot(mu_space, latent_values, marker='o')
    plt.xlabel(r'$\mu$')
    plt.ylabel(fr'$s_{component+1}(T)$')
    #plt.title(f'Latent Bifurcation Diagram for Component {component}')
    #plt.legend()
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.savefig(HyperParams.net_dir+'bifurcation_'+HyperParams.net_run+f'_latent_bifurcation_component_{component}'+'.pdf', bbox_inches='tight')
    plt.show()
    
    plt.plot(mu_space, latent_widths, color='#065895', marker='o')
    plt.xlabel(r'$\mu$')#, fontsize=18)
    plt.ylabel(fr'$A(s_{component+1})$')#, fontsize=18)
    #plt.title(f'Latent Bifurcation Diagram - Latent Width for Component {component}')
    #plt.legend()
    plt.grid(True, which='both', linestyle='--', alpha=0.1)
    plt.savefig(HyperParams.net_dir+'bifurcation_'+HyperParams.net_run+f'_variation'+'.pdf', bbox_inches='tight')
    plt.show()

    if plot_norm:
        latent_norms = np.array(latent_norms)
        plt.plot(mu_space, latent_norms, marker='o')
        plt.xlabel(r'$\mu$')
        plt.ylabel(r'$\|s(T)\|$')
        #plt.title(f'Latent Bifurcation Diagram - Latent Norm')
        #plt.legend()
        plt.grid(alpha=0.1)
        #savefig = HyperParams.plots_dir + HyperParams.net_name + HyperParams.net_run + f'_latent_bifurcation_norm.pdf'
        #plt.savefig(savefig, bbox_inches='tight')
        plt.show()

plot_bifurcation_diagram(HyperParams, results, VAR_all, mu_space, scaler_all, P_diagram)

for i in (0, 1):
    if i == 0:
        plot_flag = True
    else:
        plot_flag = False
    plot_latent_bifurcation_diagram(HyperParams, latents, mu_space, component=i, plot_norm=plot_flag)

In [ ]:
# def plot_latent_variance_diagram(HyperParams, latents, mu_space, start_time_idx, component=0):
#     """
#     Plots the variance of the latent component trajectory as a function of mu,
#     calculated over the segment starting from start_time_idx (to exclude initial transients).
#     """
#     # Assuming latents shape is (N_mu * N_times, N_latent_dims)
#     n_times = latents.shape[0] // len(mu_space)
#     latent_variances = []

#     for i in range(len(mu_space)):
#         start_index = i * n_times + start_time_idx
#         end_index = (i + 1) * n_times

#         # Check if the segment is long enough to calculate a meaningful variance (> 1 element)
#         if end_index - start_index > 1:
#             # Select the latent trajectory segment for the current mu and component, starting from start_time_idx
#             segment = latents[start_index:end_index, component]
            
#             # Calculate the variance of the segment
#             # torch.var is used here, assuming latents is a torch tensor
#             variance = torch.var(segment).item()
#             latent_variances.append(variance)
#         else:
#             # Append NaN if the segment is too short or invalid
#             latent_variances.append(np.nan) 

#     latent_variances = np.array(latent_variances)

#     plt.figure()
#     plt.plot(mu_space, latent_variances, marker='o', label=fr'$Var(s_{component+1})$')
#     plt.xlabel(r'$\mu$')
#     #plt.ylabel(fr'$Var(s_{component+1} | t \ge T_{\text{{start}}})$')
#     plt.title(f'Latent Variance Diagram (Component {component}, from index {start_time_idx})')
#     plt.legend()
#     plt.grid(alpha=0.1)
#     #savefig = HyperParams.plots_dir + HyperParams.net_name + HyperParams.net_run + f'_latent_variance_component_{component}.pdf'
#     #plt.savefig(savefig, bbox_inches='tight')
#     plt.show()

# idx = 0
# plot_latent_variance_diagram(HyperParams, latents, mu_space, start_time_idx=idx, component=0)
# plot_latent_variance_diagram(HyperParams, latents, mu_space, start_time_idx=idx, component=1)

In [ ]:
def plot_latent_norm_variance_diagram(HyperParams, latents, mu_space, start_time_idx):
    """
    Plots the variance of the norm of the latent states as a function of mu,
    calculated over the segment starting from start_time_idx (to exclude initial transients).
    """
    # Assuming latents shape is (N_mu * N_times, N_latent_dims)
    n_times = latents.shape[0] // len(mu_space)
    latent_norm_variances = []
    
    # 1. Compute the norm of the latent state vector for all time steps
    # shape is (N_mu * N_times, N_latent_dims) -> (N_mu * N_times)
    latent_norms_all = torch.norm(latents, dim=1) 

    for i in range(len(mu_space)):
        start_index = i * n_times + start_time_idx
        end_index = (i + 1) * n_times

        if end_index - start_index > 1:
            # Select the segment of norms for the current mu, starting from start_time_idx
            segment_of_norms = latent_norms_all[start_index:end_index]
            
            # Calculate the variance of the norm segment
            variance = torch.var(segment_of_norms).item()
            latent_norm_variances.append(variance)
        else:
            latent_norm_variances.append(np.nan) 

    latent_norm_variances = np.array(latent_norm_variances)

    plt.figure()
    plt.plot(mu_space, latent_norm_variances, marker='o', color='#065895')#, label=r'$Var(\|s\|)$')
    plt.xlabel(r'$\mu$')#,fontsize=18)
    plt.ylabel(r'Var$(\|s\|)$')#,fontsize=18)
    #plt.title(r'$\text{Var}$')
    #plt.legend()
    plt.grid(True, which='both', linestyle='--', alpha=0.1)
    plt.savefig(HyperParams.net_dir+'bifurcation_'+HyperParams.net_run+f'_variance'+'.pdf', bbox_inches='tight')
    plt.show()

START_TIME_IDX = 0

plot_latent_norm_variance_diagram(HyperParams, latents, mu_space, start_time_idx=START_TIME_IDX)

In [ ]:
# def compute_latent_second_derivative(latents, mu_space, dt=1.0):
#     """
#     Computes the second time derivative of the latent states using a finite 
#     difference scheme for each segment corresponding to a unique mu value.
#     """
#     total_steps, n_latent_dims = latents.shape
#     n_mu = len(mu_space)
#     n_times = total_steps // n_mu
    
#     s_double_dot = torch.zeros_like(latents)
#     dt2 = dt**2

#     for i in range(n_mu):
#         start_idx = i * n_times
#         end_idx = (i + 1) * n_times
#         s_segment = latents[start_idx:end_idx, :]
        
#         if n_times < 3:
#             s_double_dot[start_idx:end_idx, :] = 0.0
#             continue
        
#         # --- 1. Compute for Interior Points (Central Difference Scheme, O(h^2)) ---
#         s_prev = s_segment[:-2, :]
#         s_curr = s_segment[1:-1, :]
#         s_next = s_segment[2:, :]
#         s_double_dot[start_idx + 1 : end_idx - 1, :] = (s_next - 2 * s_curr + s_prev) / dt2

#         # --- 2. Boundary Point at t_0 (Index 0 - Forward Difference Scheme, O(h^2)) ---
#         if n_times >= 4:
#             s0, s1, s2, s3 = s_segment[0, :], s_segment[1, :], s_segment[2, :], s_segment[3, :]
#             s_double_dot[start_idx, :] = (2 * s0 - 5 * s1 + 4 * s2 - s3) / dt2
#         else:
#             s_double_dot[start_idx, :] = s_double_dot[start_idx + 1, :] 

#         # --- 3. Boundary Point at t_N-1 (Index N-1 - Backward Difference Scheme, O(h^2)) ---
#         if n_times >= 4:
#             sN_1 = s_segment[n_times - 1, :]
#             sN_2 = s_segment[n_times - 2, :]
#             sN_3 = s_segment[n_times - 3, :]
#             sN_4 = s_segment[n_times - 4, :]
#             s_double_dot[end_idx - 1, :] = (2 * sN_1 - 5 * sN_2 + 4 * sN_3 - sN_4) / dt2
#         else:
#             s_double_dot[end_idx - 1, :] = s_double_dot[end_idx - 2, :] 
            
#     return s_double_dot

# def plot_latent_second_derivative_time_series(s_double_dot, mu_space, mu_index, component=0):
#     """
#     Plots the time evolution of the second derivative for a specific mu value.
#     """
#     total_steps = s_double_dot.shape[0]
#     n_mu = len(mu_space)
#     n_times = total_steps // n_mu

#     if mu_index >= n_mu or mu_index < 0:
#         print(f"Error: mu_index {mu_index} is out of bounds for mu_space of size {n_mu}.")
#         return

#     start_idx = mu_index * n_times
#     end_idx = (mu_index + 1) * n_times
    
#     s_dd_timeseries = s_double_dot[start_idx:end_idx, component].cpu().numpy()
#     time = np.arange(n_times)

#     plt.figure()
#     plt.plot(time, s_dd_timeseries, label=fr'$\mu={mu_space[mu_index]:.4f}$')
#     plt.xlabel('Time Step Index')
#     #plt.ylabel(fr'$s''_{component+1}(t)$')
#     plt.title(f'Time Series of Latent Second Derivative (Component {component})')
#     plt.legend()
#     plt.grid(alpha=0.1)
#     plt.show()

# # --- NEW FUNCTIONS ---

# def plot_latent_norm_time_series(latents, mu_space):
#     """
#     Plots the time evolution of the latent state norm (||s(t)||) for ALL mu values 
#     on a single plot to show dynamic changes.
#     """
#     total_steps = latents.shape[0]
#     n_mu = len(mu_space)
#     n_times = total_steps // n_mu
    
#     # Calculate the norm for all time steps
#     latent_norms_all = torch.norm(latents, dim=1).cpu().numpy() 
#     time = np.arange(n_times)
    
#     plt.figure(figsize=(10, 6))
#     plt.title(r'Time Evolution of Latent State Norm $\|s(t)\|$ for all $\mu$')
#     plt.xlabel('Time Step Index')
#     plt.ylabel(r'$\|s(t)\|$')
#     plt.grid(alpha=0.3)
    
#     # Use a colormap to distinguish trajectories
#     cmap = plt.get_cmap('viridis')
#     colors = [cmap(i) for i in np.linspace(0, 1, n_mu)]

#     for i in range(n_mu):
#         start_idx = i * n_times
#         end_idx = (i + 1) * n_times
#         norm_timeseries = latent_norms_all[start_idx:end_idx]
        
#         plt.plot(time, norm_timeseries, color=colors[i], 
#                  label=fr'$\mu={mu_space[i]:.4f}$', linewidth=1)

#     # Add a color bar or simplify the legend for clarity if n_mu is large
#     if n_mu <= 15:
#         plt.legend(title=r'$\mu$ Values', loc='upper right', fontsize='small')
#     else:
#         print(f"Too many mu values ({n_mu}) to show individual labels. Showing plot without full legend.")
        
#     plt.show()


# def plot_latent_second_derivative_norm_time_series(s_double_dot, mu_space):
#     """
#     Plots the time evolution of the second derivative norm (||s''(t)||) for ALL mu values 
#     on a single plot.
#     """
#     total_steps = s_double_dot.shape[0]
#     n_mu = len(mu_space)
#     n_times = total_steps // n_mu
    
#     # Calculate the norm of the second derivative for all time steps
#     s_dd_norms_all = torch.norm(s_double_dot, dim=1).cpu().numpy()
#     time = np.arange(n_times)
    
#     plt.figure(figsize=(10, 6))
#     #plt.title(r'Time Evolution of Second Derivative Norm $\|\mathbf{s}''(t)\|$ for all $\mu$')
#     plt.xlabel('Time Step Index')
#     #plt.ylabel(r'$\|\mathbf{s}''(t)\|$')
#     plt.grid(alpha=0.3)
    
#     # Use a colormap to distinguish trajectories
#     cmap = plt.get_cmap('plasma')
#     colors = [cmap(i) for i in np.linspace(0, 1, n_mu)]

#     for i in range(n_mu):
#         start_idx = i * n_times
#         end_idx = (i + 1) * n_times
#         norm_timeseries = s_dd_norms_all[start_idx:end_idx]
        
#         plt.plot(time, norm_timeseries, color=colors[i], 
#                  label=fr'$\mu={mu_space[i]:.4f}$', linewidth=1)

#     # Add a color bar or simplify the legend for clarity if n_mu is large
#     if n_mu <= 15:
#         plt.legend(title=r'$\mu$ Values', loc='upper right', fontsize='small')
#     else:
#         print(f"Too many mu values ({n_mu}) to show individual labels. Showing plot without full legend.")

#     plt.show()


# # --- Execution Block ---

# # Set a typical start index to exclude transient behavior
# START_TIME_IDX = 0

# # Compute the second derivative of the latent states
# DT_STEP = 0.5 # Placeholder for time step size
# s_double_dot = compute_latent_second_derivative(latents, mu_space, dt=DT_STEP)


# # 2. Plot the time evolution of the second derivative norm (requested)
# plot_latent_second_derivative_norm_time_series(s_double_dot, mu_space)

# # Final 1D bifurcation plot for the variance of the norm
# #plot_latent_norm_variance_diagram(HyperParams, latents, mu_space, start_time_idx=START_TIME_IDX)

# def plot_max_latent_second_derivative_norm_diagram(s_double_dot, mu_space):
#     """
#     Plots the maximum value of the second derivative norm (max_t ||s''(t)||) 
#     for each mu value, creating a max-norm bifurcation diagram for acceleration.
#     """
#     total_steps = s_double_dot.shape[0]
#     n_mu = len(mu_space)
#     n_times = total_steps // n_mu
    
#     max_s_dd_norms = []
    
#     # 1. Calculate the norm of the second derivative for all time steps
#     # shape is (N_mu * N_times, N_latent_dims) -> (N_mu * N_times)
#     s_dd_norms_all = torch.norm(s_double_dot, dim=1) 

#     # 2. Iterate through each mu segment and find the maximum norm value
#     for i in range(n_mu):
#         start_idx = i * n_times
#         end_idx = (i + 1) * n_times
        
#         # Select the segment of norms for the current mu
#         segment_of_norms = s_dd_norms_all[start_idx:end_idx]
        
#         # Find the maximum value in this segment
#         max_norm = torch.max(segment_of_norms).item()
#         max_s_dd_norms.append(max_norm)

#     max_s_dd_norms = np.array(max_s_dd_norms)

#     # 3. Plotting
#     plt.figure(figsize=(8, 5))
#     plt.plot(mu_space, max_s_dd_norms, marker='o', linestyle='-', color='red')
#     plt.xlabel(r'$\mu$')
#     plt.ylabel(r'$\max_t \|\mathbf{s}''(t)\|$')
#     plt.title(r'Max Norm of Latent Second Derivative vs $\mu$')
#     plt.grid(alpha=0.3)
#     plt.show()


# plot_max_latent_second_derivative_norm_diagram(s_double_dot, mu_space)


# import matplotlib.pyplot as plt
# import numpy as np
# def plot_latent_phase_space(HyperParams, latents, mu_space):
#     """
#     Plots the phase space trajectory of latent components s1 vs s2 for 
#     each value of the bifurcation parameter mu.
    
#     Args:
#         HyperParams (object): Object containing simulation parameters (e.g., for saving).
#         latents (Tensor): The latent trajectory data (N_total_snapshots x N_latent_components).
#         mu_space (list/array): List of parameter values mu.
#     """
#     # -----------------------------------------------------------
#     # 1. Setup Data Extraction
#     # -----------------------------------------------------------
#     n_mu = len(mu_space)
#     if latents.dim() != 2:
#         print("Error: Latents tensor must be 2D (N_total_snapshots x N_latent_components).")
#         return

#     n_latent_components = latents.shape[1]
#     if n_latent_components < 2:
#         print("Error: Need at least 2 latent components (s1, s2) for phase space plotting.")
#         return
        
#     n_times_total = latents.shape[0]
#     n_times_per_mu = n_times_total // n_mu

#     # Check if data length is evenly divisible
#     if n_times_total % n_mu != 0:
#         print(f"Warning: Total snapshots ({n_times_total}) not perfectly divisible by number of mu values ({n_mu}). Using floor division.")

#     # -----------------------------------------------------------
#     # 2. Plotting Setup
#     # -----------------------------------------------------------
#     fig, ax = plt.subplots(figsize=(8, 8))
    
#     # Generate colors for each mu value
#     cmap = plt.cm.get_cmap('viridis', n_mu)

#     # -----------------------------------------------------------
#     # 3. Loop and Plot Trajectories
#     # -----------------------------------------------------------
#     for i in range(n_mu):
#         mu = mu_space[i]
        
#         # Define the start and end indices for the current mu's trajectory
#         start_idx = i * n_times_per_mu
#         end_idx = (i + 1) * n_times_per_mu
        
#         # Extract s1 and s2 trajectories
#         s1_trajectory = latents[start_idx:end_idx, 0].cpu().numpy()
#         s2_trajectory = latents[start_idx:end_idx, 1].cpu().numpy()
        
#         # Plot the trajectory: s2 vs s1
#         # Use a low alpha for the line and a marker for the end point
#         ax.plot(s1_trajectory, s2_trajectory, 
#                 color=cmap(i), 
#                 linewidth=1.5, 
#                 alpha=0.6,
#                 #label=fr'$\mu={mu:.2f}$'
#                 )
        
#         # Plot the starting point (optional, helpful for understanding direction)
#         ax.plot(s1_trajectory[0], s2_trajectory[0], 
#                 color=cmap(i), 
#                 marker='^', 
#                 markersize=6, 
#                 markeredgecolor='black', 
#                 label='_nolegend_')

#         # Plot the end point (final steady/periodic state)
#         ax.plot(s1_trajectory[-1], s2_trajectory[-1], 
#                 color=cmap(i), 
#                 marker='o', 
#                 markersize=6, 
#                 markeredgecolor='black', 
#                 label='_nolegend_')

#     # -----------------------------------------------------------
#     # 4. Finalize Plot
#     # -----------------------------------------------------------
#     ax.set_xlabel(fr'$s_1$ (Latent Component 0)')
#     ax.set_ylabel(fr'$s_2$ (Latent Component 1)')
#     ax.set_title('Latent Phase Space Trajectories $s_1$ vs $s_2$')
#     ax.legend(loc='best', title=r'Parameter $\mu$')
#     ax.grid(True, alpha=0.3)
#     ax.set_aspect('equal', adjustable='box') # Keep scales consistent for phase space

#     # Save and show
#     # savefig = HyperParams.plots_dir + HyperParams.net_name + HyperParams.net_run + '_latent_phase_space.pdf'
#     # plt.savefig(savefig, bbox_inches='tight')
#     plt.show()

# plot_latent_phase_space(HyperParams, latents, mu_space)

# import matplotlib.pyplot as plt
# import numpy as np
# from mpl_toolkits.mplot3d import Axes3D # Import for 3D plotting
# # Assuming torch is available

# def plot_latent_phase_space_3d(HyperParams, latents, mu_space, dt=0.01, azim=0, elev=0):
#     """
#     Plots the 3D trajectory of latent components s1 vs s2 vs Time (t) 
#     for each value of the bifurcation parameter mu.
    
#     Args:
#         HyperParams (object): Object containing simulation parameters.
#         latents (Tensor): The latent trajectory data (N_total_snapshots x N_latent_components).
#         mu_space (list/array): List of parameter values mu.
#         dt (float): The time step size used in the simulation. Used to generate the time axis.
#     """
#     # -----------------------------------------------------------
#     # 1. Setup Data Extraction
#     # -----------------------------------------------------------
#     n_mu = len(mu_space)
#     if latents.dim() != 2 or latents.shape[1] < 2:
#         print("Error: Latents tensor must be 2D with at least 2 components.")
#         return

#     n_times_total = latents.shape[0]
#     n_times_per_mu = n_times_total // n_mu
    
#     if n_times_total % n_mu != 0:
#         print(f"Warning: Total snapshots ({n_times_total}) not perfectly divisible by number of mu values ({n_mu}). Using floor division.")

#     # -----------------------------------------------------------
#     # 2. Plotting Setup
#     # -----------------------------------------------------------
#     fig = plt.figure(figsize=(10, 8))
#     # Create a 3D subplot
#     ax = fig.add_subplot(111, projection='3d')
#     ax.view_init(elev=elev, azim=azim)
    
#     # Generate colors for each mu value
#     cmap = plt.cm.get_cmap('plasma', n_mu)

#     # -----------------------------------------------------------
#     # 3. Loop and Plot Trajectories
#     # -----------------------------------------------------------
    
#     for i in range(n_mu):
#         mu = mu_space[i]
        
#         # Define the start and end indices for the current mu's trajectory
#         start_idx = i * n_times_per_mu
#         end_idx = (i + 1) * n_times_per_mu
        
#         # Extract s1 and s2 trajectories
#         s1_trajectory = latents[start_idx:end_idx, 0].cpu().numpy()
#         s2_trajectory = latents[start_idx:end_idx, 1].cpu().numpy()
        
#         # Generate the time axis for this trajectory
#         time_points = np.arange(0, len(s1_trajectory) * dt, dt)
        
#         if len(time_points) > len(s1_trajectory):
#              time_points = time_points[:len(s1_trajectory)]

#         # Plot the 3D trajectory: (s1, s2, t)
#         ax.plot(s1_trajectory, s2_trajectory, time_points, 
#                 color=cmap(i), 
#                 linewidth=1.5, 
#                 alpha=0.8,
#                 #label=fr'$\mu={mu:.2f}$'
#                 )
        
#         # Plot the starting point (optional: helps see the initial transient)
#         ax.scatter(s1_trajectory[0], s2_trajectory[0], time_points[0], 
#                    color=cmap(i), 
#                    marker='^', 
#                    s=40, 
#                    edgecolors='black', 
#                    label='_nolegend_')

#         # Plot the end point (final steady state or limit cycle point)
#         ax.scatter(s1_trajectory[-1], s2_trajectory[-1], time_points[-1], 
#                    color=cmap(i), 
#                    marker='o', 
#                    s=40, 
#                    edgecolors='black', 
#                    label='_nolegend_')


#     # -----------------------------------------------------------
#     # 4. Finalize Plot
#     # -----------------------------------------------------------
#     ax.set_xlabel(fr'$s_1$')
#     ax.set_ylabel(fr'$s_2$')
#     ax.set_zlabel(r'Time $t$')
#     ax.set_title('Latent Trajectories in Phase-Time Space ($s_1, s_2, t$)')
#     ax.legend(loc='best', title=r'Parameter $\mu$')
#     ax.grid(True, alpha=0.5)

#     # Save and show
#     # savefig = HyperParams.plots_dir + HyperParams.net_name + HyperParams.net_run + '_latent_phase_time_space_3d.pdf'
#     # plt.savefig(savefig, bbox_inches='tight')
#     plt.show()

# plot_latent_phase_space_3d(HyperParams, latents, mu_space, 0.5, -35, 40)

In [ ]:
def plot_latent_total_variation(HyperParams, latents, mu_space, component=0, plot_norm=False):
    """
    Calculates and plots the total variation of the latent space for each mu value.
    Total Variation (TV) is defined as the sum of absolute differences between consecutive
    time steps for the specified component or the L2 norm of the latent vector.
    
    Args:
        HyperParams: Object containing hyperparameters (e.g., plots_dir).
        latents (torch.Tensor): Latent space trajectory. Shape (N_mu * N_times, N_components).
        mu_space (list or np.array): List of mu parameter values.
        component (int): The index of the latent component to analyze (default 0).
        plot_norm (bool): If True, plots the total variation of the L2 norm of the latent vector.
    """
    n_times = latents.shape[0] // len(mu_space)
    total_variations = []
    
    # Determine the index range for each mu block
    mu_indices = [(i * n_times, (i + 1) * n_times) for i in range(len(mu_space))]

    for start_idx, end_idx in mu_indices:
        # Get the time series for the current mu
        current_latents = latents[start_idx:end_idx, :]
        
        if plot_norm:
            # Calculate the L2 norm for each time step in this block
            #norms = torch.norm(current_latents, dim=1)
            # Calculate the absolute difference between consecutive norms
            #diffs = torch.abs(norms[1:] - norms[:-1])
            diffs = current_latents[1:] - current_latents[:-1]
            norms = torch.norm(diffs, dim =1)
            # Sum the differences for the Total Variation
            #total_variation = torch.sum(diffs).item()
            total_variation = torch.sum(norms).item()
            y_label = r'$TV(\|s\|) = \sum |\|s(t+1)\| - \|s(t)\||$'
        else:
            # Get the specified component time series
            component_series = current_latents[:, component]
            # Calculate the absolute difference between consecutive component values
            diffs = torch.abs(component_series[1:] - component_series[:-1])
            # Sum the differences for the Total Variation
            total_variation = torch.sum(diffs).item()
            y_label = fr'$TV(s_{component}) = \sum |s_{component}(t+1) - s_{component}(t)|$'
            
        total_variations.append(total_variation)

    # --- Plotting ---
    plt.plot(mu_space, total_variations, marker='o', linestyle='-')
    plt.xlabel(r'$\mu$')
    plt.ylabel(y_label)
    #plt.title('Latent Total Variation Diagram')
    plt.grid(alpha=0.1)
    # savefig = HyperParams.plots_dir + HyperParams.net_name + HyperParams.net_run + f'_latent_total_variation_comp_{component}_norm_{plot_norm}.pdf'
    # plt.savefig(savefig, bbox_inches='tight')
    plt.show()



for i in (0, 1):
    # Plot Total Variation for latent components s0 and s1
    plot_latent_total_variation(HyperParams, latents, mu_space, component=i, plot_norm=False)

# Plot Total Variation for the latent norm ||s|| (using component=0 just as a placeholder)
plot_latent_total_variation(HyperParams, latents, mu_space, component=0, plot_norm=True)